# 2) Cargar viajes en un dataframe y zonas en otro.

In [1]:
from pyspark.sql import functions as F
B = "s3://taxis-gotys"

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1791650341562_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
viajes = spark.read.parquet(f"{B}/taxis/viajes/")
viajes.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

In [3]:
zonas = spark.read.csv(f"{B}/taxis/zonas/", header=True, inferSchema=True)
zonas.show(5, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 5 rows

In [5]:
print("Viajes:", viajes.count())
print("Zonas:", zonas.count())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Viajes: 102871387
Zonas: 265

# 3) Realizar un join para para realizar una consulta donde se liste: Lugar de recogida, hora_recogida, hora_dejada, DOLOCATION_ID(usar un alias para Lugar de recogida, PU_ZONE) 

In [6]:
zonas_pu = zonas.select(
    F.col("LocationID").alias("PU_ID"),
    F.col("Zone").alias("PU_ZONE")
)

consulta1 = viajes.join(
    F.broadcast(zonas_pu),
    viajes.PULocationID == zonas_pu.PU_ID
).select(
    "PU_ZONE",
    F.col("tpep_pickup_datetime").alias("hora_recogida"),
    F.col("tpep_dropoff_datetime").alias("hora_dejada"),
    "DOLocationID"
)

consulta1.show(10, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------------------+-------------------+-------------------+------------+
|PU_ZONE                       |hora_recogida      |hora_dejada        |DOLocationID|
+------------------------------+-------------------+-------------------+------------+
|Long Island City/Hunters Point|2018-03-01 00:01:34|2018-03-01 00:01:43|145         |
|Manhattan Valley              |2018-03-01 00:14:34|2018-03-01 00:28:13|244         |
|Upper West Side North         |2018-03-01 00:51:25|2018-03-01 00:59:54|152         |
|Long Island City/Hunters Point|2018-03-01 00:00:01|2018-03-01 00:00:17|145         |
|Long Island City/Hunters Point|2018-03-01 00:55:10|2018-03-01 00:56:36|145         |
|Long Island City/Hunters Point|2018-03-01 00:56:55|2018-03-01 00:57:01|145         |
|Queensbridge/Ravenswood       |2018-03-01 00:43:30|2018-03-01 00:50:40|193         |
|Queensbridge/Ravenswood       |2018-03-01 00:52:10|2018-03-01 00:52:21|193         |
|Astoria                       |2018-03-01 00:13:21|20

# 4) A partir de la anterior consulta realizar, una segunda consulta que cruce nuevamente y de el nombre de zona de dejada: PU_ZONE, DO_ZONE, PU_datetime, DO_datetime. 

In [7]:
zonas_do = zonas.select(
    F.col("LocationID").alias("DO_ID"),
    F.col("Zone").alias("DO_ZONE")
)

consulta2 = consulta1.join(
    F.broadcast(zonas_do),
    consulta1.DOLocationID == zonas_do.DO_ID
).select(
    "PU_ZONE",
    "DO_ZONE",
    F.col("hora_recogida").alias("PU_datetime"),
    F.col("hora_dejada").alias("DO_datetime")
)

consulta2.show(10, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------------------+------------------------------+-------------------+-------------------+
|PU_ZONE                       |DO_ZONE                       |PU_datetime        |DO_datetime        |
+------------------------------+------------------------------+-------------------+-------------------+
|Long Island City/Hunters Point|Long Island City/Hunters Point|2018-03-01 00:01:34|2018-03-01 00:01:43|
|Manhattan Valley              |Washington Heights South      |2018-03-01 00:14:34|2018-03-01 00:28:13|
|Upper West Side North         |Manhattanville                |2018-03-01 00:51:25|2018-03-01 00:59:54|
|Long Island City/Hunters Point|Long Island City/Hunters Point|2018-03-01 00:00:01|2018-03-01 00:00:17|
|Long Island City/Hunters Point|Long Island City/Hunters Point|2018-03-01 00:55:10|2018-03-01 00:56:36|
|Long Island City/Hunters Point|Long Island City/Hunters Point|2018-03-01 00:56:55|2018-03-01 00:57:01|
|Queensbridge/Ravenswood       |Queensbridge/Ravenswood       |2

# 5)Crear una nueva columna trip_duration que sea la resta entre DO_datetime y PU_datetime. Para esto pasar a unixtimestamp las columnas. El resultado debe quedar en minutos. a) withColumn (crear columna) https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.DataFrame.withColumn.html y b) Pasar a unix_timestamp https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.functions.unix_timestamp.html

In [8]:
df = consulta2.withColumn(
    "trip_duration",
    F.round(
        (F.unix_timestamp("DO_datetime") - F.unix_timestamp("PU_datetime")) / 60,
        2
    )
)

df.cache()
df.show(10, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------------------+------------------------------+-------------------+-------------------+-------------+
|PU_ZONE                       |DO_ZONE                       |PU_datetime        |DO_datetime        |trip_duration|
+------------------------------+------------------------------+-------------------+-------------------+-------------+
|Long Island City/Hunters Point|Long Island City/Hunters Point|2018-03-01 00:01:34|2018-03-01 00:01:43|0.15         |
|Manhattan Valley              |Washington Heights South      |2018-03-01 00:14:34|2018-03-01 00:28:13|13.65        |
|Upper West Side North         |Manhattanville                |2018-03-01 00:51:25|2018-03-01 00:59:54|8.48         |
|Long Island City/Hunters Point|Long Island City/Hunters Point|2018-03-01 00:00:01|2018-03-01 00:00:17|0.27         |
|Long Island City/Hunters Point|Long Island City/Hunters Point|2018-03-01 00:55:10|2018-03-01 00:56:36|1.43         |
|Long Island City/Hunters Point|Long Island City/Hunters

# 6) Mostrar el Viaje que más ha durado

In [9]:
print("Viaje más largo (datos crudos):")
df.orderBy(F.desc("trip_duration")).show(1, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Viaje m?s largo (datos crudos):
+--------------------+--------+-------------------+-------------------+-------------+
|PU_ZONE             |DO_ZONE |PU_datetime        |DO_datetime        |trip_duration|
+--------------------+--------+-------------------+-------------------+-------------+
|TriBeCa/Civic Center|Kips Bay|2018-11-09 08:06:08|2018-11-29 08:43:06|28836.97     |
+--------------------+--------+-------------------+-------------------+-------------+
only showing top 1 row

In [12]:
ZONAS_INVALIDAS = ["NV", "N/A", "Unknown", "Outside of NYC"]

limpio = df.filter(
    (F.year("PU_datetime") == 2018)
    & (F.col("trip_duration") > 0)
    & (F.col("trip_duration") <= 24 * 60)
    & F.col("PU_ZONE").isNotNull()
    & F.col("DO_ZONE").isNotNull()
    & ~F.col("PU_ZONE").isin(ZONAS_INVALIDAS)
    & ~F.col("DO_ZONE").isin(ZONAS_INVALIDAS)
)
limpio.cache()

print("Viaje mas largo (datos limpios):")
limpio.orderBy(F.desc("trip_duration")).show(1, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Viaje mas largo (datos limpios):
+-------------------+-----------+-------------------+-------------------+-------------+
|PU_ZONE            |DO_ZONE    |PU_datetime        |DO_datetime        |trip_duration|
+-------------------+-----------+-------------------+-------------------+-------------+
|UN/Turtle Bay South|JFK Airport|2018-10-26 15:33:02|2018-10-27 15:33:01|1439.98      |
+-------------------+-----------+-------------------+-------------------+-------------+
only showing top 1 row

# 7) Determinar cuál es el destino más frecuente. Excluir posibles datos erróneos. 

In [13]:
destinos = limpio.groupBy("DO_ZONE").count().orderBy(F.desc("count"))
destinos.show(10, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------------------+-------+
|DO_ZONE                     |count  |
+----------------------------+-------+
|Upper East Side North       |3970775|
|Midtown Center              |3893729|
|Upper East Side South       |3713646|
|Murray Hill                 |3274321|
|Times Sq/Theatre District   |3170874|
|Midtown East                |3150683|
|Clinton East                |2805386|
|Union Sq                    |2766729|
|Lincoln Square East         |2741615|
|Penn Station/Madison Sq West|2686531|
+----------------------------+-------+
only showing top 10 rows

# 8) Determinar cuál es el viaje más frecuente(combinación origen-destino), y cuál el tiempo promedio de viaje. Incluir Desviación estándar. 

In [14]:
rutas = (
    limpio.groupBy("PU_ZONE", "DO_ZONE")
    .agg(
        F.count("*").alias("num_viajes"),
        F.round(F.avg("trip_duration"), 2).alias("duracion_promedio_min"),
        F.round(F.stddev("trip_duration"), 2).alias("desviacion_std_min"),
    )
    .orderBy(F.desc("num_viajes"))
)

rutas.show(10, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------------------+---------------------+----------+---------------------+------------------+
|PU_ZONE              |DO_ZONE              |num_viajes|duracion_promedio_min|desviacion_std_min|
+---------------------+---------------------+----------+---------------------+------------------+
|Upper East Side South|Upper East Side North|610448    |8.03                 |49.55             |
|Upper East Side North|Upper East Side North|515517    |5.88                 |45.83             |
|Upper East Side North|Upper East Side South|512924    |9.85                 |52.11             |
|Upper East Side South|Upper East Side South|482837    |6.68                 |48.15             |
|Upper West Side South|Upper West Side North|290759    |6.51                 |48.32             |
|Upper West Side South|Lincoln Square East  |285398    |7.73                 |53.02             |
|Upper East Side South|Midtown East         |270977    |10.0                 |53.71             |
|Lincoln Square East

In [16]:
print("Ruta mas frecuente con origen distinto al destino:")
rutas.filter(F.col("PU_ZONE") != F.col("DO_ZONE")).show(10, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Ruta mas frecuente con origen distinto al destino:
+---------------------+---------------------+----------+---------------------+------------------+
|PU_ZONE              |DO_ZONE              |num_viajes|duracion_promedio_min|desviacion_std_min|
+---------------------+---------------------+----------+---------------------+------------------+
|Upper East Side South|Upper East Side North|610448    |8.03                 |49.55             |
|Upper East Side North|Upper East Side South|512924    |9.85                 |52.11             |
|Upper West Side South|Upper West Side North|290759    |6.51                 |48.32             |
|Upper West Side South|Lincoln Square East  |285398    |7.73                 |53.02             |
|Upper East Side South|Midtown East         |270977    |10.0                 |53.71             |
|Lincoln Square East  |Upper West Side South|270257    |8.15                 |52.64             |
|Upper East Side South|Midtown Center       |251922    |12.81      

In [17]:
ruta_top = limpio.filter(
    (F.col("PU_ZONE") == "Upper East Side South")
    & (F.col("DO_ZONE") == "Upper East Side North")
)

ruta_top.select(
    F.count("*").alias("n"),
    F.round(F.avg("trip_duration"), 2).alias("promedio"),
    F.round(F.stddev("trip_duration"), 2).alias("desviacion"),
    F.round(F.expr("percentile_approx(trip_duration, 0.5)"), 2).alias("mediana"),
    F.round(F.expr("percentile_approx(trip_duration, 0.99)"), 2).alias("p99"),
    F.round(F.max("trip_duration"), 2).alias("maximo"),
).show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+--------+----------+-------+-----+-------+
|     n|promedio|desviacion|mediana|  p99| maximo|
+------+--------+----------+-------+-----+-------+
|610448|    8.03|     49.55|   5.73|16.68|1439.72|
+------+--------+----------+-------+-----+-------+

# 9 y 10) Crear columnas enteras year y month

In [22]:
df = df.withColumn("year", F.year("PU_datetime").cast("int")) \
       .withColumn("month", F.month("PU_datetime").cast("int"))

df.printSchema()
df.select("PU_datetime", "year", "month").show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- PU_ZONE: string (nullable = true)
 |-- DO_ZONE: string (nullable = true)
 |-- PU_datetime: timestamp_ntz (nullable = true)
 |-- DO_datetime: timestamp_ntz (nullable = true)
 |-- trip_duration: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

+-------------------+----+-----+
|        PU_datetime|year|month|
+-------------------+----+-----+
|2018-03-01 00:01:34|2018|    3|
|2018-03-01 00:14:34|2018|    3|
|2018-03-01 00:51:25|2018|    3|
|2018-03-01 00:00:01|2018|    3|
|2018-03-01 00:55:10|2018|    3|
+-------------------+----+-----+
only showing top 5 rows

# 11) Listar los meses con número de viajes ordenados de mayor a menor 

In [23]:
(
    df.filter(F.col("year") == 2018)
    .groupBy("year", "month")
    .count()
    .withColumnRenamed("count", "num_viajes")
    .orderBy(F.desc("num_viajes"))
    .show(12)
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+-----+----------+
|year|month|num_viajes|
+----+-----+----------+
|2018|    3|   9430359|
|2018|    4|   9305987|
|2018|    5|   9224816|
|2018|   10|   8834553|
|2018|    1|   8760903|
|2018|    6|   8714545|
|2018|    2|   8493198|
|2018|   12|   8194900|
|2018|   11|   8156023|
|2018|    9|   8048894|
|2018|    8|   7854947|
|2018|    7|   7850980|
+----+-----+----------+

# 12) Guardar el dataset que incluye PU_ZONE, DO_ZONE, PU_datetime, DO_datetime, trip_duration, year, month en un bucket en S3 en formato parquet. Particionar los datos por year y month. La estructura debe quedar: s3://bucket/viajes/year=xxxx/month=xxxx 

In [24]:
(
    df.filter(F.col("year") == 2018)
    .select("PU_ZONE", "DO_ZONE", "PU_datetime", "DO_datetime",
            "trip_duration", "year", "month")
    .repartition("year", "month")
    .write.mode("overwrite")
    .partitionBy("year", "month")
    .parquet(f"{B}/viajes/")
)

print("Guardado en", f"{B}/viajes/")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Guardado en s3://taxis-gotys/viajes/